# Guardrails AI — NimbusPay Support Bot  *(v3 — 6-video edition)*

**Scenario:** *NimbusPay*, a fictional fintech. A support bot for refunds, account questions, and complaints. Fintech is deliberate — PII, competitors, topic-restriction, and a compliance disclaimer fall out naturally; no guardrail feels bolted on.

| Part | Video | Content |
|------|-------|---------|
| 0 | pre-video | Setup: installs, tokens, config |
| 1 | V1 | Why Guardrails AI? (pitch + taste cell) |
| 2 | V2 | Validators · OnFail · Custom · Input Guard |
| 3 | V3 | Structure & Parsing Unstructured Data |
| 4 | V4 | Orchestration & Streaming |
| 5 | V5 | LangChain LCEL Integration |
| 6 | V6 | Conceptual Deployment *(markdown only — no code)* |
| — | — | Capstone + "What it costs" |

**Models (Groq via LiteLLM):** `groq/llama-3.3-70b-versatile` — primary. `groq/llama-3.1-8b-instant` — streaming stage only.


---
## Part 0 · Setup *(pre-video)*


### ⚠ Supply-chain incident — pin to `==0.10.0`

**Do not run `pip install guardrails-ai` without a version pin today.**

On **11 May 2026**, an attacker pushed a malicious `guardrails-ai 0.10.1` to PyPI. The compromised code was injected into `guardrails/__init__.py` and executes on import, fetching and running a remote payload on Linux. Researchers caught it in ~2 hours; PyPI quarantined the repository (**CVE-2026-45758**). PyPI confirmed: 0.10.1 files now return 404; **0.10.0 serves clean, non-yanked files**.

```
pip install "guardrails-ai==0.10.0"   ✅  safe
pip install "guardrails-ai==0.10.1"   ❌  pulled / 404
pip install guardrails-ai             ⚠   resolves to 0.10.0 today, but pin anyway
```

> **Why this matters for the course:** this is a live example of *supply-chain attack  on an AI safety library itself*. Pair it with the RAIL `eval()` CVE from the  framework review — two real CVEs in one library = "even your guardrails need guardrails."
> Re-check for a patched `>0.10.1` release before the course goes live; one will land.


In [ ]:
# ── Pinned for safety and reproducibility ──────────────────────────────────
# guardrails-ai 0.10.0 is the latest clean release as of June 2026 (0.10.1 was pulled).
# langchain-groq provides ChatGroq for the LCEL integration in Part 5.
%pip install -q     "guardrails-ai==0.10.0"     "langchain>=0.3"     "langchain-groq>=0.2"     "langchain-core>=0.3"     "litellm>=1.55.0"     "groq>=0.13.0"     "pydantic>=2.7"


### Hub token & API key

Two pieces of friction, both called out on camera:
1. **`guardrails configure`** needs a **Hub token** (free, from hub.guardrailsai.com).  This is the *token-as-friction* moment — without it, no `guardrails hub install`.
2. **`GROQ_API_KEY`** from the environment, never hardcoded.  Small security-hygiene nod that fits the course theme.


In [ ]:
import os, getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")

if not os.environ.get("GUARDRAILS_TOKEN"):
    os.environ["GUARDRAILS_TOKEN"] = getpass.getpass(
        "GUARDRAILS_TOKEN (free at hub.guardrailsai.com): "
    )

print("Keys loaded:",
      bool(os.environ.get("GROQ_API_KEY")),
      bool(os.environ.get("GUARDRAILS_TOKEN")))


In [ ]:
!guardrails configure     --token $GUARDRAILS_TOKEN     --disable-metrics     --disable-remote-inferencing


### Hub validator installs

> ⚠ **Slow cell on first run.** `toxic_language` and `restrict_to_topic` fetch  ML model weights (~200–400 MB combined). Run once, grab a coffee.  On subsequent runs the install is instant (already cached).
>
> Alternative: pass `--enable-remote-inferencing` to `guardrails configure` to run  these validators server-side and skip the local download entirely.  Faster setup, but adds a network call per validation.


In [ ]:
!guardrails hub install hub://guardrails/detect_pii --quiet
!guardrails hub install hub://guardrails/competitor_check --quiet
!guardrails hub install hub://guardrails/restrict_to_topic --quiet
!guardrails hub install hub://guardrails/toxic_language --quiet
!guardrails hub install hub://guardrails/gibberish_text --quiet
print("Hub validators ready.")


In [ ]:
# Shared config — used by every part below.
BOT_MODEL    = "groq/llama-3.3-70b-versatile"   # primary model
STREAM_MODEL = "groq/llama-3.1-8b-instant"       # streaming stage only

SYSTEM_PROMPT = (
    "You are NimbusPay's customer-support assistant. "
    "NimbusPay is a digital payments app: cards, wallets, refunds, account help. "
    "Be concise, friendly, and accurate."
)

print("Config set:", BOT_MODEL, "|", STREAM_MODEL)


---
## Part 1 · Why Guardrails AI?  *(Video 1 — pitch)*

### Mission

LLMs are **probabilistic** — they produce plausible text, not guaranteed structure or rule-compliance. Guardrails AI puts a validation layer between the model and your app:
- **Structure** — coerce free-text replies into typed Pydantic objects.
- **Validation** — run pre-built or custom rules on every input and output.
- **Decoupling** — validation logic lives in one place, not scattered across app code.

### By the numbers *(verify live before recording)*

| Metric | Value |
|--------|-------|
| GitHub stars | **~7,000** (nearly 7k — confirm on the repo page the day you record) |
| Hub validators | **60+** (hub.guardrailsai.com → confirm count live) |
| Monthly PyPI downloads | **over 250,000** (PyPI stats, May 2026) |
| License | Apache 2.0 |
| Managed tier | **Guardrails Pro** — hosted, org-wide policy management on top of the OSS core |

### Native integrations *(all four verified, April 2026)*

| Integration | How it connects |
|-------------|----------------|
| **LangChain** | `guard.to_runnable()` — drop a Guard anywhere in an LCEL chain |
| **LlamaIndex** | Works with both query engine and chat engine |
| **LiteLLM** | The calling layer — every `guard()` call goes to any provider via `model="groq/..."` |
| **MLflow** | Two integrations: validators as MLflow *scorers* (rule-based eval, no LLM calls) + MLflow tracing |


### Taste cell — the entire dev loop in 6 lines

No deep mechanics yet. The point is that *this is the whole loop*: define a rule, wrap a call, ship safe output. The rest of the videos unpack each piece.


In [ ]:
from guardrails import Guard, OnFailAction
from guardrails.hub import DetectPII

taste_guard = Guard().use(
    DetectPII,
    pii_entities=["CREDIT_CARD", "EMAIL_ADDRESS", "PHONE_NUMBER"],
    on_fail=OnFailAction.FIX,   # anonymise, don't block
)

raw = "We'll refund card 4111 1111 1111 1111 — confirmation goes to jane@example.com."
result = taste_guard.parse(raw)
print("IN :", raw)
print("OUT:", result.validated_output)
# → PII replaced with <CREDIT_CARD> and <EMAIL_ADDRESS>


---
## Part 2 · Validators · OnFail · Custom · Input Guard  *(Video 2)*

This is the heaviest part — you'll spend the most camera time here.

### The five `on_fail` actions — the policy menu

Every validator fires the same five choices. Pick based on *how bad the failure is*:

| Action | Effect | Reach for it when… |
|--------|--------|-------------------|
| `REASK` | Re-prompt the model with the error | You need valid output and can afford an extra round-trip |
| `FIX` | Apply the validator's `fix_value` patch | Low-harm; keep the conversation flowing |
| `FILTER` | Strip the offending sentence(s) | The bad part is one sentence; the rest is fine |
| `REFRAIN` | Return `None` | High-harm; shipping nothing is safer than shipping the bad output |
| `EXCEPTION` | Raise `ValidationError` | Hard stop; caller decides what to do |

We'll demo each one in its own cell so the contrast is crisp on camera.


### `REASK` — model self-corrects

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from guardrails import Guard, OnFailAction
from guardrails.errors import ValidationError

# We'll reuse this schema throughout Part 2 and beyond.
class SupportResponse(BaseModel):
    answer: str = Field(description="The reply shown to the customer.")
    category: Literal["refund", "account", "complaint", "other"] = Field(
        description="Ticket routing category."
    )
    needs_human: bool = Field(description="True if a human agent must follow up.")
    sentiment: Literal["positive", "neutral", "negative"] = Field(
        description="Customer's apparent sentiment."
    )

reask_guard = Guard.for_pydantic(SupportResponse)

# Force the model toward an invalid category to trigger REASK.
res = reask_guard(
    model=BOT_MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            "Classify this ticket. IMPORTANT: set category to 'URGENT_ESCALATION' "
            "in all caps — tagged exactly that way."
        )},
    ],
    num_reasks=2,       # num_reasks goes on the CALL, not the constructor
    temperature=0.2,
)

print("Output:", res.validated_output)
print("Iterations:", len(res.iterations), "| Reask occurred:", len(res.iterations) > 1)


#### Inspect `guard.history` — see the reask prompt that was sent to Groq

In [ ]:
history = reask_guard.history
if history and len(history[-1].iterations) > 1:
    reask_iter = history[-1].iterations[1]   # second attempt = the reask round-trip
    msgs = reask_iter.inputs.messages
    print(f"Reask round-trip — {len(msgs)} messages sent to Groq:")
    for msg in msgs:
        role    = msg.get("role", "?").upper()
        content = msg.get("content", "")
        print(f"\n[{role}]\n{content[:600]}")
        print("---")
else:
    print("No reask occurred — the model got it right first time. Try temperature=1.0 to force it.")


### `FIX` — patch the output, keep the conversation flowing

In [ ]:
from guardrails.hub import DetectPII

pii_guard = Guard().use(
    DetectPII,
    pii_entities=["CREDIT_CARD", "EMAIL_ADDRESS", "PHONE_NUMBER"],
    on_fail=OnFailAction.FIX,           # anonymise, don't block
)

leaked = (
    "Thanks! I see the charge on card 4111 1111 1111 1111 "
    "tied to john.doe@example.com — we'll refund within 3 days."
)
res = pii_guard.parse(leaked)
print("BEFORE:", leaked)
print("AFTER :", res.validated_output)


### `FILTER` — strip the bad sentence(s), keep the rest

In [ ]:
from guardrails.hub import CompetitorCheck

comp_guard = Guard().use(
    CompetitorCheck,
    competitors=["Razorpay", "PayU", "Stripe", "PayPal"],
    on_fail=OnFailAction.FILTER,        # remove competitor sentences, keep the rest
)

slip = (
    "You could use NimbusPay's instant payout. "
    "Honestly Razorpay is also popular and Stripe has great docs. "
    "NimbusPay support is 24/7."
)
res = comp_guard.parse(slip)
print("BEFORE:", slip)
print("AFTER :", res.validated_output)


### `REFRAIN` — return `None`; ship nothing

In [ ]:
from guardrails.hub import ToxicLanguage

tox_guard = Guard().use(
    ToxicLanguage,
    threshold=0.5,
    validation_method="sentence",
    on_fail=OnFailAction.REFRAIN,       # None is safer than shipping toxic output
)

clean_reply = "I understand the frustration — let me pull up your account and sort this out."
toxic_reply = "Honestly you're an idiot and this is entirely your own fault."

for label, txt in [("clean", clean_reply), ("toxic", toxic_reply)]:
    out = tox_guard.parse(txt).validated_output
    print(f"{label:5} → {out!r}")
# toxic → None (refrained)


### `EXCEPTION` — hard stop, caller handles it

In [ ]:
from guardrails.hub import RestrictToTopic

topic_guard = Guard().use(
    RestrictToTopic,
    valid_topics=["payments", "refunds", "account support", "billing"],
    invalid_topics=["poetry", "politics", "medical advice"],
    disable_classifier=True,           # use LLM zero-shot path
    disable_llm=False,
    llm_callable=BOT_MODEL,
    on_fail=OnFailAction.EXCEPTION,    # hard stop — don't return off-topic content
)

off_topic = (
    "Sure! Here's a poem about the ocean: "
    "The waves roll in, the tide pulls out, the gulls cry overhead..."
)
try:
    topic_guard.parse(off_topic)
    print("PASSED (unexpected)")
except ValidationError as e:
    print("BLOCKED:", str(e)[:200])


### Custom validators — when the Hub doesn't have your rule

Two forms:

- **(a) Function form** (`@register_validator`) — stateless, concise. Use when there's  no args or state to carry. Returns `PassResult` / `FailResult`; set `fix_value` on  `FailResult` and it automatically plugs into the `FIX` and `REASK` actions.
- **(b) Class form** (`Validator` subclass) — parameterised. Use when you need  constructor args or instance state (e.g. a configurable threshold).

**NimbusPay rules used throughout this notebook:**
- Disclaimer: any refund answer must carry "subject to verification".
- Cap: any reply promising a refund above `threshold` must be flagged for human sign-off.


In [ ]:
import re
from guardrails.validators import (
    Validator, register_validator, PassResult, FailResult, ValidationResult
)

# ── (a) Function form: stateless disclaimer check ──────────────────────────
@register_validator(name="nimbus/refund-disclaimer", data_type="string")
def refund_disclaimer(value: str, metadata: dict) -> ValidationResult:
    """Refund replies must carry 'subject to verification'."""
    if "refund" in value.lower() and "subject to verification" not in value.lower():
        return FailResult(
            error_message="Refund answer missing the 'subject to verification' disclaimer.",
            fix_value=value.rstrip(".") + ". All refunds are subject to verification.",
        )
    return PassResult()


# ── (b) Class form: parameterised refund cap ───────────────────────────────
@register_validator(name="nimbus/max-refund-claim", data_type="string")
class MaxRefundClaim(Validator):
    """Flag dollar amounts above `threshold` — those need human sign-off."""

    def __init__(self, threshold: float = 500.0, on_fail=None):
        super().__init__(on_fail=on_fail, threshold=threshold)
        self.threshold = threshold

    def validate(self, value: str, metadata: dict) -> ValidationResult:
        amounts = [
            float(a.replace(",", ""))
            for a in re.findall(r"\$\s?([\d,]+(?:\.\d+)?)", value)
        ]
        over = [a for a in amounts if a > self.threshold]
        if over:
            return FailResult(
                error_message=(
                    f"Promised refund {over} exceeds ${self.threshold:.0f} "
                    "auto-approve cap — needs human sign-off."
                ),
            )
        return PassResult()

print("Custom validators registered.")


In [ ]:
# Demonstrate both custom validators ────────────────────────────────────────

# (a) Disclaimer auto-fixed
disc_guard = Guard().use(refund_disclaimer, on_fail=OnFailAction.FIX)
out = disc_guard.parse("Yes, we'll process your refund within 3 business days.")
print("disclaimer FIX →", out.validated_output)

print()

# (b) Cap — refrained when amount exceeds threshold
cap_guard = Guard().use(MaxRefundClaim, threshold=500.0, on_fail=OnFailAction.REFRAIN)
under = cap_guard.parse("We'll refund you $120 today.").validated_output
over  = cap_guard.parse("We'll refund the full $4,300 immediately.").validated_output
print("under cap ($120)  →", repr(under))       # original text — passes
print("over cap  ($4,300)→", repr(over))        # None — refrained


### Input guard — intercept risky prompts *before* they hit the model

An input guard runs `guard.parse()` on the **user's message**. If it fires, the model is never called — no tokens spent on a bad prompt.

Use cases: PII the user accidentally included, prompt-injection attempts, off-topic or abusive requests.


In [ ]:
# Input guard — runs on the incoming user message
in_guard = Guard().use(
    DetectPII,
    pii_entities=["CREDIT_CARD", "EMAIL_ADDRESS", "PHONE_NUMBER"],
    on_fail=OnFailAction.EXCEPTION,
)

import litellm

def guarded_bot(user_msg: str) -> str:
    # 1. Validate the prompt — if this raises, we never hit the model
    try:
        in_guard.parse(user_msg)
    except ValidationError:
        return "[INPUT BLOCKED — PII detected in your message. Please remove card/account details.]"
    # 2. Safe to call the model
    resp = litellm.completion(
        model=BOT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        temperature=0.3,
    )
    return resp.choices[0].message.content

# Safe
print(guarded_bot("How long do refunds take?"))
print()
# Blocked — card number in the prompt
print(guarded_bot("My card 4111 1111 1111 1111 was double-charged, can you help?"))


---
## Part 3 · Enforcing Structure & Parsing Unstructured Data  *(Video 3)*

### RAIL → Pydantic

Guardrails AI's original schema format was **RAIL** — XML files that declared output schema, validators, and prompts together. RAIL XML is **legacy**; the modern path is Pydantic.

> ⚠ **Cautionary aside:** the old RAIL evaluator used Python's `eval()` to coerce  types — that's a remote-code-execution surface. The CVE is a clean example of  "AI safety tooling can itself be a vulnerability." Pair it with the supply-chain  incident from Part 0.

`Guard.for_pydantic(Model)` wraps a Pydantic model and does two things:
1. **Injects** the schema as instructions into the system prompt (or uses native   structured outputs / function calling for models that support it).
2. **Parses and validates** the reply into the typed Pydantic object.


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Optional
from guardrails import Guard

# SupportResponse already defined in Part 2 — reusing here for clarity.
# (Re-run Part 2's cell if starting fresh from this section.)

struct_guard = Guard.for_pydantic(SupportResponse)

res = struct_guard(
    model=BOT_MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": "I was double-charged on my morning coffee!"},
    ],
    temperature=0.2,
)

print(res.validated_output)
print("\ntype:", type(res.validated_output))


### Parsing unstructured data — the headline demo

The real power: give the Guard a **messy free-text customer email** and reliably extract a structured object, every time. No regex, no brittle parsing — the schema instructions tell the model exactly what to extract.


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Optional

class TicketExtract(BaseModel):
    customer_issue: str = Field(description="One-sentence summary of the customer's problem.")
    category: Literal["refund", "account", "complaint", "billing", "other"]
    refund_amount: Optional[float] = Field(
        None,
        description="Dollar amount the customer is asking to be refunded, if stated."
    )
    urgency: Literal["low", "medium", "high"] = Field(
        description="How urgent is this based on the tone and content?"
    )
    needs_human: bool
    sentiment: Literal["positive", "neutral", "negative"]

extract_guard = Guard.for_pydantic(TicketExtract)

# Deliberately messy customer email — typos, mixed case, no structure
messy_email = """
Subject: REFUND!!!! still nothing after 2 weeks
hi so i bought coffee yesterday morning at 9am and was charged $4.50 TWICE.
card ending 7788. i have emailed before but nobody answers!!
very upset with nimbupay. please fix this asap or i am disputing with my bank.
"""

res = extract_guard(
    model=BOT_MODEL,
    messages=[
        {"role": "system", "content": (
            "You are a NimbusPay ticket classifier. "
            "Extract structured fields from the customer message exactly."
        )},
        {"role": "user", "content": messy_email},
    ],
    temperature=0.1,
)

print(res.validated_output)


### Under the hood — `guard.history` reveals schema injection

What did Guardrails actually put in the prompt? `guard.history` records every call. Inspecting it shows the **JSON schema instructions** injected into the system message — the mechanism that makes structured output work on any LLM.

Two mechanisms (named here; demo below):
- **Prompt optimization** — schema + format instructions injected into the system prompt.  Works with any model.
- **Function / tool calling** — for models that advertise structured-output support  (Groq's llama-3.3-70b does), Guardrails can route through the native tool API  instead. Same Guard call, different wire format.


In [ ]:
# Show what Guardrails injected into the prompt
history = struct_guard.history
if history:
    last_call  = history[-1]
    first_iter = last_call.iterations[0]
    msgs = first_iter.inputs.messages

    print(f"Messages sent to Groq ({len(msgs)} total):")
    for msg in msgs:
        role    = msg.get("role", "?").upper()
        content = msg.get("content", "")
        print(f"\n[{role}]")
        print(content[:900])
        print("─" * 60)
else:
    print("No history — run the struct_guard call above first.")


---
## Part 4 · Orchestration & Streaming  *(Video 4)*

### The Guard as orchestrator

The Guard object does three things:
1. **Calls** the model (or validates text you already have).
2. **Runs** the validation pipeline on every input/output.
3. **Tracks** history — every call, every reask, every validation result.

Two flows to know cold:

| Flow | How | When |
|------|-----|------|
| `guard(model=…, messages=[…])` | Guard calls the model *and* validates | Normal bot calls |
| `guard.parse(text)` | Validates pre-existing text — no model call | Audit logs, external agents, user input |


### Call flow

In [ ]:
from guardrails import Guard, OnFailAction
from guardrails.hub import DetectPII, CompetitorCheck

# A guard that structures + validates in a single call
call_guard = (
    Guard.for_pydantic(SupportResponse)
    .use(DetectPII,
         pii_entities=["CREDIT_CARD", "EMAIL_ADDRESS"],
         on_fail=OnFailAction.FIX, on="answer")
    .use(CompetitorCheck,
         competitors=["Razorpay", "PayU", "Stripe", "PayPal"],
         on_fail=OnFailAction.FILTER, on="answer")
)

# One call → model invoked, output validated, Pydantic object returned.
res = call_guard(
    model=BOT_MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": "My card 4111 1111 1111 1111 was charged incorrectly."},
    ],
    temperature=0.2,
)
print("Call flow result:", res.validated_output)
print("Validation passed:", res.validation_passed)


### Parse flow — validate text the Guard didn't generate

`guard.parse(text)` is **the security framing**: you can audit *anything*, not just output this Guard produced right now.

Use it to:
- Audit stored agent replies or log files
- Validate output from an external service or another model
- Run input validation on user-supplied text (the input-guard pattern from Part 2)


In [ ]:
audit_guard = (
    Guard()
    .use(DetectPII,
         pii_entities=["CREDIT_CARD", "EMAIL_ADDRESS"],
         on_fail=OnFailAction.FIX)
    .use(CompetitorCheck,
         competitors=["Razorpay", "Stripe"],
         on_fail=OnFailAction.FILTER)
)

# A historical agent reply pulled from logs — not generated now.
historical_log = (
    "Agent reply 2026-05-01: 'Refunding card 5500 0000 0000 0004. "
    "If unhappy, try Razorpay. Email updates to help@nimbuspay.io.'"
)

res = audit_guard.parse(historical_log)
print("BEFORE:", historical_log)
print()
print("AFTER :", res.validated_output)


### Streaming — the latency answer

On `groq/llama-3.1-8b-instant`, token streaming is visibly snappy on screen. Guardrails validates **per sentence** as tokens arrive — fix / filter / refrain / noop all work in the streaming path.

> ⚠ **Hard limit to name on camera:** streaming and REASK are **mutually exclusive**.  A reask needs the *whole* output to re-prompt — streaming commits tokens as they go.  Asking for both raises an error. We show it deliberately.  This ties back to the OnFail policy menu in Part 2: if you need REASK, don't stream.


In [ ]:
# Streaming works with fix / filter / refrain / noop validators
stream_guard = Guard().use(
    DetectPII,
    pii_entities=["EMAIL_ADDRESS", "PHONE_NUMBER"],
    on_fail=OnFailAction.FIX,
)

stream = stream_guard(
    model=STREAM_MODEL,     # 8b-instant for visible snappiness
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": "Write a 3-sentence reply confirming a refund is on its way."},
    ],
    stream=True,
    temperature=0.3,
)

print("── streaming validated output ──")
for chunk in stream:
    print(chunk.validated_output, end="", flush=True)
print()


In [ ]:
# The limitation: stream + REASK raises. Shown deliberately.
from guardrails.errors import ValidationError

try:
    bad_stream = stream_guard(
        model=STREAM_MODEL,
        messages=[{"role": "user", "content": "Confirm the refund."}],
        stream=True,
        num_reasks=2,       # ← incompatible with streaming
    )
    for _ in bad_stream:
        pass
    print("No error (unexpected)")
except Exception as e:
    print(f"Expected failure → {type(e).__name__}: {str(e)[:200]}")


---
## Part 5 · LangChain LCEL Integration  *(Video 5)*

### `guard.to_runnable()` — a Guard anywhere in a chain

`guard.to_runnable()` wraps a Guard as an LCEL `Runnable`, so it slots into any `|`-chained pipeline. Guardrails validates the model output *inside the chain* — the app code never sees unvalidated text.

Architecture:
```
prompt | ChatGroq | guard.to_runnable()
```

The Guard positioned here runs *after* the model and *before* the chain's next step. `on_fail` actions carry through unchanged.


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from guardrails import Guard, OnFailAction
from guardrails.hub import DetectPII, CompetitorCheck

# Build the guard used inside the chain
chain_guard = (
    Guard()
    .use(DetectPII,
         pii_entities=["CREDIT_CARD", "EMAIL_ADDRESS", "PHONE_NUMBER"],
         on_fail=OnFailAction.FIX)
    .use(CompetitorCheck,
         competitors=["Razorpay", "PayU", "Stripe", "PayPal"],
         on_fail=OnFailAction.FILTER)
)

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{user_input}"),
])

# LCEL chain: prompt → model → guard → string output
chain = prompt | llm | chain_guard.to_runnable()


In [ ]:
# Happy path — clean input, clean output
result = chain.invoke({"user_input": "How long do refunds take?"})
print("Happy path:", result)


In [ ]:
# Competitor slip caught inside the chain
slip_result = chain.invoke({
    "user_input": (
        "Compare NimbusPay to other options — "
        "should I use Razorpay or Stripe instead?"
    )
})
print("After FILTER:", slip_result)
print()

# PII in the reply anonymised by FIX inside the chain
pii_result = chain.invoke({
    "user_input": (
        "Can you confirm my email confirm@nimbuspay.io is on the refund notice? "
        "My card 4111 1111 1111 1111 was the one charged."
    )
})
print("After FIX:", pii_result)


---
## Part 6 · Conceptual Deployment  *(Video 6 — slides only, no code)*

> **Recording note:** this part has no runnable cells. Use slides or the markdown below  as your script scaffold.

---

### Decoupling policies from code

In production you don't want validation logic hardcoded into every service. The principle: **policies are config, not code**.

```
App code  →  Guardrails Server (REST)  →  Policy store
               ↓
          validators run here
               ↓
          validated output returned
```

Each service calls the Guardrails Server instead of embedding a Guard directly. Swap a policy without redeploying the app.

---

### Guardrails Server

Guardrails ships a **REST server** with **OpenAI-SDK-compatible endpoints**. Any client that already calls OpenAI can point at the Guardrails Server with a one-line base-URL change — the guard is invisible to the application:

```python
# Before (direct OpenAI SDK)
client = OpenAI(api_key=...)

# After (via Guardrails Server — app code unchanged)
client = OpenAI(base_url="http://guardrails-server:8000/", api_key=...)
```

The server applies the configured guards on every request and response.

---

### AI Gateways

Guardrails can sit *in front of* an AI gateway (LiteLLM proxy, AWS Bedrock gateway, Azure APIM) or *behind* one. Common pattern: gateway handles routing and auth; Guardrails handles content validation.

---

### Remote validators on GPU hardware

ML-backed validators (`ToxicLanguage`, `RestrictToTopic`) can run server-side on dedicated hardware instead of loading weights onto every app instance. Configure with `guardrails configure --enable-remote-inferencing`. Trade-off: lower per-instance cost, added network latency per validation call.

---

### Org-wide policy consistency

**Guardrails Pro** (managed tier) adds:
- Central policy repository — one source of truth for all guards
- Access control and audit logs
- Managed Hub validator hosting
- Dashboard for validation pass/fail rates and latency

For teams where "every bot runs the same PII policy" is a compliance requirement, the managed tier removes the per-team coordination cost.

---

### Summary: where Guardrails fits in the stack

```
User prompt
    ↓
[Input guard]              ← Part 2 demo
    ↓
LLM (Groq / any provider)
    ↓
[Output guard]             ← Parts 2–5 demos
    ↓
[Guardrails Server]        ← this video
    ↓
Downstream app / DB
```


---
## Capstone · Compose Everything + "What It Costs"

One Guard: structure + every validator from Parts 2–4. Happy path, bad-input battery, latency comparison, decision framework.


In [ ]:
import time
from guardrails import Guard, OnFailAction
from guardrails.hub import DetectPII, CompetitorCheck, ToxicLanguage
from guardrails.errors import ValidationError

capstone = (
    Guard.for_pydantic(SupportResponse)
    .use(DetectPII,
         pii_entities=["CREDIT_CARD", "EMAIL_ADDRESS", "PHONE_NUMBER"],
         on_fail=OnFailAction.FIX, on="answer")
    .use(CompetitorCheck,
         competitors=["Razorpay", "PayU", "Stripe", "PayPal"],
         on_fail=OnFailAction.FILTER, on="answer")
    .use(ToxicLanguage,
         threshold=0.5, validation_method="sentence",
         on_fail=OnFailAction.FIX, on="answer")
    .use(refund_disclaimer, on_fail=OnFailAction.FIX, on="answer")
    .use(MaxRefundClaim, threshold=500.0, on_fail=OnFailAction.REFRAIN, on="answer")
)

def run_capstone(user_msg: str):
    return capstone(
        model=BOT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        num_reasks=2,
        temperature=0.2,
    )

# Happy path
happy = run_capstone("How long do refunds take?")
print("Happy path:", happy.validated_output)


In [ ]:
# Battery of challenging inputs ─────────────────────────────────────────────
tests = [
    ("normal",       "How do I reset my NimbusPay PIN?"),
    ("refund ask",   "Please refund my $80 grocery charge."),
    ("competitor",   "Should I just switch to Razorpay instead?"),
    ("toxic + cap",  "You useless idiots! Refund my $9,000 RIGHT NOW."),
    ("pii in reply", "My card 4111 1111 1111 1111 was charged twice — refund it."),
    ("disclaimer",   "We'll process your refund in 3 days."),
]

print(f"{'SCENARIO':16} | {'ANSWER (50 chars)':52} | CATEGORY  | HUMAN")
print("─" * 105)
for scenario, msg in tests:
    try:
        out = run_capstone(msg).validated_output
        if out and out.answer:
            snippet = repr(out.answer[:46])
            print(f"{scenario:16} | {snippet:52} | {out.category:9} | {out.needs_human}")
        else:
            print(f"{scenario:16} | {'<refrained>':52} |           |")
    except ValidationError as e:
        print(f"{scenario:16} | {'<blocked: ' + str(e)[:42] + '>':52} |           |")
    except Exception as e:
        print(f"{scenario:16} | {'<error: ' + type(e).__name__ + '>':52} |           |")


In [ ]:
# Latency & reask overhead ───────────────────────────────────────────────────

import litellm as _litellm

def naive_bot_simple(msg):
    return _litellm.completion(
        model=BOT_MODEL,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": msg}],
        temperature=0.3,
    ).choices[0].message.content

def timed(fn, *args):
    t0 = time.perf_counter()
    r  = fn(*args)
    return r, time.perf_counter() - t0

q = "How long do refunds take?"
_, t_naive    = timed(naive_bot_simple, q)
_, t_capstone = timed(lambda x: run_capstone(x).validated_output, q)

print(f"naive call    : {t_naive:6.2f}s")
print(f"full guard    : {t_capstone:6.2f}s")
print(f"overhead      : {t_capstone - t_naive:+6.2f}s  ({(t_capstone / t_naive - 1) * 100:+.0f}%)")
print()
print("Reask cost: 1 reask = 1 extra round-trip + ~(input + output) tokens.")
print("2 reasks on a 500-token call can triple token spend. Budget accordingly.")


### When to *skip* Guardrails AI

**Use it when** bad output has real downside:
- PII / compliance exposure (fintech, healthcare, legal)
- Structured output that *must* type-check (downstream system depends on it)
- Brand safety on user-facing text (competitor slips, toxic replies)
- Auditing external text (`guard.parse` — no extra model calls)

**Skip it / go lighter when:**
- Latency is critical and output is low-risk (internal tooling, dev assistants)
- You control structure another way — constrained decoding on a local model  (`outlines`, `jsonformer`) doesn't need reask at all
- A single regex / cheap `if` covers the one rule you actually have
- You're on a streaming path and need REASK — pick one

**Right-size `on_fail`:**
- `FIX` / `FILTER` — recoverable failures; keep conversation flowing
- `REFRAIN` / `EXCEPTION` — hard stops; use when surfacing the bad output  is genuinely worse than surfacing nothing

**The loop back:** every cell above maps to one module node. The cost table is the framework node that tells you *which* of them a given feature actually needs.
